<a href="https://colab.research.google.com/github/da-luce/cs5782_final_project/blob/sennet-branch/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/da-luce/cs5782_final_project.git
%cd cs5782_final_project
!git checkout sennet-branch

In [ ]:
!pip install -r requirements.txt

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# Run #1: full fine-tuning baseline on SST-2 (~10-15 min on a T4).
# Defaults: 10k train samples, full validation, 5 epochs, batch 32, lr 2e-5.
!python code/train.py --mode baseline --dataset sst2

In [ ]:
# Run #2: LoRA on SST-2 (~10-15 min on a T4).
# Same data/epochs/batch as the baseline above; only LR differs (5e-4, per the paper).
!python code/train.py --mode lora --dataset sst2

In [ ]:
# Run #3: full fine-tuning baseline on MNLI (~15-20 min on a T4).
!python code/train.py --mode baseline --dataset mnli

In [ ]:
# Run #4: LoRA on MNLI (~15-20 min on a T4).
!python code/train.py --mode lora --dataset mnli

In [ ]:
import json
import os

import pandas as pd

info_dir = "results/info"
rows = []
for fname in sorted(os.listdir(info_dir)):
    if not fname.endswith(".json"):
        continue
    with open(os.path.join(info_dir, fname)) as f:
        info = json.load(f)

    for prefix, result in info["eval_results"].items():
        rows.append({
            "mode": info["mode"],
            "dataset": info["dataset"],
            "split": prefix,
            "accuracy": round(result.get(f"{prefix}_accuracy", float("nan")), 4),
            "trainable": info["param_report"]["trainable"],
            "lora_params": info["param_report"]["lora_trainable"],
            "train_size": info["train_size"],
            "epochs": info["epochs"],
            "time_min": round(info["training_time_sec"] / 60, 1),
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))